# M6: Request State Machine

In M5 a request's lifecycle was implicit. You could tell where it was only by which queue held it and whether `finish_reason` was still `None`. M6 makes the lifecycle explicit: `Req.status` is a small state machine, and every change goes through `set_status`.

```text
WAITING ──get_next_batch_to_run──▶ RUNNING ──process_batch_result──▶ FINISHED   (EOS or max_new_tokens)
                                           └─────────────────────────▶ FAILED     (run_batch raised)
```

Why a request stopped is now an object, named after SGLang's classes:

| Class | When | `to_json()` |
|---|---|---|
| `FINISH_MATCHED_TOKEN` | last token is one of `eos_token_ids` | `{"type": "stop", "matched": <id>}` |
| `FINISH_LENGTH` | `len(output_ids) >= max_new_tokens` | `{"type": "length", "length": <n>}` |
| `FINISH_ERROR` | the forward step raised | `{"type": "error", "message": ...}` (not in SGLang) |

This notebook checks:

- A new `Req` is `WAITING`; only legal transitions are allowed, and terminal states have no way out
- `update_finish_state`: EOS wins over length, and either of Qwen3's two EOS tokens stops
- Watch real requests move through the states while the scheduler runs
- `meta_info.finish_reason` in the HTTP response is now an object
- A failing forward step ends in `FAILED` with `FINISH_ERROR`, and the server keeps serving

## Setup

Same as M5: start uvicorn in a background thread so the notebook can act as the client.

Port 30000 is SGLang's default. Stop any `python server.py` you started in a terminal first, or the port is taken.

In [1]:
import json
import threading
import time

import httpx
import uvicorn

from engine import Engine
from req import FINISH_LENGTH, FINISH_MATCHED_TOKEN, Req, ReqStatus
from server import create_app

engine = Engine()
runner = engine.model_runner
tok = engine.tokenizer
print(runner.device, "| eos_token_ids:", {i: tok.decode([i]) for i in runner.eos_token_ids})

server = uvicorn.Server(uvicorn.Config(create_app(engine), host="127.0.0.1", port=30000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
while not server.started:
    time.sleep(0.1)

client = httpx.Client(base_url="http://127.0.0.1:30000", timeout=300)


def show(resp):
    print(resp.status_code, resp.reason_phrase)
    try:
        print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
    except json.JSONDecodeError:
        print(repr(resp.text))  # not every error body is JSON

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

mps | eos_token_ids: {151643: '<|endoftext|>', 151645: '<|im_end|>'}


## Two EOS Tokens

Qwen3's `generation_config.json` lists two EOS ids: `<|im_end|>` ends a chat turn, and `<|endoftext|>` ends a document. The tokenizer's `eos_token_id` is only one of them. If we stopped only on `tok.eos_token_id`, a model that ends with the other token would keep generating until `max_new_tokens`.

`ModelRunner` reads the full list from `generation_config`. `Engine` copies it into every `Req.eos_token_ids`, as SGLang's `Req` does.

In [2]:
print("generation_config.eos_token_id:", runner.model.generation_config.eos_token_id)
print("tokenizer.eos_token_id        :", tok.eos_token_id, repr(tok.eos_token))

generation_config.eos_token_id: [151645, 151643]
tokenizer.eos_token_id        : 151645 '<|im_end|>'


## Transitions by Hand

Build a `Req` directly. It is never submitted, so no other thread touches it. `_TRANSITIONS` in `req.py` is the whole state machine: each status maps to the set of statuses it may move to.

In [3]:
from req import _TRANSITIONS

for src, dsts in _TRANSITIONS.items():
    print(f"{src.name:<8} -> {sorted(d.name for d in dsts) or '(terminal)'}")

req = Req(rid="manual", prompt="", input_ids=[1], max_new_tokens=4, eos_token_ids=runner.eos_token_ids)
print("\nnew Req:", req.status.name, "| finished():", req.finished())
req.set_status(ReqStatus.RUNNING)
req.set_status(ReqStatus.FINISHED)
print("after RUNNING, FINISHED:", req.status.name)

WAITING  -> ['RUNNING']
RUNNING  -> ['FAILED', 'FINISHED']
FINISHED -> (terminal)
FAILED   -> (terminal)

new Req: WAITING | finished(): False
after RUNNING, FINISHED: FINISHED


Illegal moves raise instead of silently changing state. Each one below would be a scheduler bug: skipping `RUNNING`, going back to the queue, or reviving a finished request.

In [4]:
def try_path(*path):
    r = Req(rid="t", prompt="", input_ids=[1], max_new_tokens=4)
    try:
        for status in path:
            r.set_status(status)
        print("ok      ", " -> ".join(s.name for s in path))
    except RuntimeError as e:
        print("rejected", " -> ".join(s.name for s in path), f"| {e} | status stays {r.status.name}")


W, R, F, X = ReqStatus.WAITING, ReqStatus.RUNNING, ReqStatus.FINISHED, ReqStatus.FAILED
try_path(R, X)
try_path(F)
try_path(R, W)
try_path(R, F, R)
try_path(R, X, F)

ok       RUNNING -> FAILED
rejected FINISHED | illegal transition ReqStatus.WAITING -> ReqStatus.FINISHED | status stays WAITING
rejected RUNNING -> WAITING | illegal transition ReqStatus.RUNNING -> ReqStatus.WAITING | status stays RUNNING
rejected RUNNING -> FINISHED -> RUNNING | illegal transition ReqStatus.FINISHED -> ReqStatus.RUNNING | status stays FINISHED
rejected RUNNING -> FAILED -> FINISHED | illegal transition ReqStatus.FAILED -> ReqStatus.FINISHED | status stays FAILED


## `update_finish_state`: EOS First, Then Length

The scheduler appends a token, then asks the request whether it is done. Order matters when the last allowed token is also EOS: checking EOS first reports `stop`, which says the model chose to end. `length` would wrongly say it was cut off.

In [5]:
im_end, endoftext = tok.convert_tokens_to_ids(["<|im_end|>", "<|endoftext|>"])


def finish(output_ids, max_new_tokens):
    r = Req(rid="t", prompt="", input_ids=[1], max_new_tokens=max_new_tokens, eos_token_ids=runner.eos_token_ids)
    r.output_ids = list(output_ids)
    r.update_finish_state()
    return r.finished_reason.to_json() if r.finished() else None


print("mid generation      :", finish([11], max_new_tokens=4))
print("<|im_end|>          :", finish([11, im_end], max_new_tokens=4))
print("<|endoftext|>       :", finish([11, endoftext], max_new_tokens=4))
print("hit max_new_tokens  :", finish([11, 12], max_new_tokens=2))
print("EOS on the last step:", finish([11, im_end], max_new_tokens=2))

mid generation      : None
<|im_end|>          : {'type': 'stop', 'matched': 151645}
<|endoftext|>       : {'type': 'stop', 'matched': 151643}
hit max_new_tokens  : {'type': 'length', 'length': 2}
EOS on the last step: {'type': 'stop', 'matched': 151645}


## Watch Requests Move Through the States

Submit two `Req`s straight to the scheduler, without HTTP, so the notebook holds the same objects the scheduler thread changes. Then sample `status` and token count every 50 ms and print only the changes.

Capacity is still one, so B sits in `WAITING` for as long as A is `RUNNING`. `RUNNING` lasts many scheduler steps: a new token does not change the status, it just grows `output_ids`.

In [6]:
import uuid


def make(prompt, max_new_tokens):
    return Req(
        rid=uuid.uuid4().hex,
        prompt=prompt,
        input_ids=tok.encode(prompt),
        max_new_tokens=max_new_tokens,
        eos_token_ids=runner.eos_token_ids,
    )


a = make("A: Write a story about a dragon.", 24)
b = make("B: Write a poem about the sea.", 8)
engine.scheduler.submit(a)
engine.scheduler.submit(b)

start, last = time.perf_counter(), None
while not (a.done.is_set() and b.done.is_set()):
    now = (a.status.name, b.status.name)
    if now != last:
        print(f"{time.perf_counter() - start:5.2f}s  A={now[0]:<8} ({len(a.output_ids):>2} tok)  B={now[1]:<8} ({len(b.output_ids)} tok)")
        last = now
    time.sleep(0.05)
print(f"{time.perf_counter() - start:5.2f}s  A={a.status.name:<8} ({len(a.output_ids):>2} tok)  B={b.status.name:<8} ({len(b.output_ids)} tok)")
print("A:", a.finished_reason.to_json(), "| B:", b.finished_reason.to_json())

 0.00s  A=WAITING  ( 0 tok)  B=WAITING  (0 tok)
 0.06s  A=RUNNING  ( 0 tok)  B=WAITING  (0 tok)
 1.42s  A=FINISHED (24 tok)  B=RUNNING  (0 tok)
 1.74s  A=FINISHED (24 tok)  B=FINISHED (8 tok)
A: {'type': 'length', 'length': 24} | B: {'type': 'length', 'length': 8}


## `finish_reason` over HTTP

The response now carries the finish reason object, as SGLang's does. A raw prompt usually runs to the limit and gets `length`. A chat-templated question ends its turn with `<|im_end|>` and gets `stop`, with the matched id. `enable_thinking=False` skips Qwen3's `<think>` block.

In [7]:
resp = client.post("/generate", json={"text": "The capital of France is", "sampling_params": {"max_new_tokens": 8}})
print(resp.json()["meta_info"]["finish_reason"])

chat = tok.apply_chat_template(
    [{"role": "user", "content": "Reply with one word: what color is the sky?"}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
resp = client.post("/generate", json={"text": chat, "sampling_params": {"max_new_tokens": 64}})
show(resp)
matched = resp.json()["meta_info"]["finish_reason"]["matched"]
print("matched:", matched, repr(tok.decode([matched])))

{'type': 'length', 'length': 8}
200 OK
{
  "text": "sky",
  "output_ids": [
    26684,
    151645
  ],
  "meta_info": {
    "id": "7d4189687cf24ed4be2ab9c5e4d71034",
    "finish_reason": {
      "type": "stop",
      "matched": 151645
    },
    "prompt_tokens": 23,
    "completion_tokens": 2
  }
}
matched: 151645 '<|im_end|>'


## A Failing Step Ends in `FAILED`

Make `forward` raise. `event_loop` catches the exception and passes it to `process_batch_result` as `result`. The request gets `FINISH_ERROR` and moves `RUNNING -> FAILED`, then `running_req` is cleared so the next request can run.

First submit a `Req` directly to see the object, then go through HTTP: `Engine.generate` raises the stored error, and FastAPI turns it into a `500`.

In [8]:
def failing_forward(req):
    raise RuntimeError("model crashed")


runner.forward = failing_forward

bad = make("Hello", 8)
engine.scheduler.submit(bad)
bad.done.wait()
print(bad.status.name, bad.finished_reason.to_json(), "| output_ids:", bad.output_ids)

# uvicorn drops the connection after an unhandled error; close it so the next request opens a fresh one
show(client.post("/generate", json={"text": "Hello"}, headers={"Connection": "close"}))

del runner.forward  # back to the real model
show(client.post("/generate", json={"text": "Hello", "sampling_params": {"max_new_tokens": 8}}))

FAILED {'type': 'error', 'message': 'model crashed'} | output_ids: []
500 Internal Server Error
'Internal Server Error'


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/uvicorn/protocols/http/h11_impl.py", line 411, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/starlette/applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/Users/amikai/Workspace/mini-rt/.venv/lib/python3.11/site-packages/starlette/middl

200 OK
{
  "text": " Answer! I'm a bit confused about",
  "output_ids": [
    21806,
    0,
    358,
    2776,
    264,
    2699,
    21815,
    911
  ],
  "meta_info": {
    "id": "ce5bd4558e8b4fcab01752f393fd7edc",
    "finish_reason": {
      "type": "length",
      "length": 8
    },
    "prompt_tokens": 1,
    "completion_tokens": 8
  }
}


A `FAILED` request can have empty `output_ids`: it failed on its very first step. A `FINISHED` request always has at least one token, because `max_new_tokens >= 1` is checked before submit.

**Something to think about.** `set_status` raises on an illegal move, but `process_batch_result` runs outside `event_loop`'s `try`. So a state machine bug kills the scheduler thread, and every waiting handler hangs. Should a scheduler bug fail only one request, or crash the whole process? SGLang lets the scheduler process crash.

## Summary

| | M5 | M6 |
|---|---|---|
| Where a request is in its life | implied by which queue holds it | `Req.status`: `WAITING` / `RUNNING` / `FINISHED` / `FAILED` |
| Who changes it | scattered assignments | only `set_status`, checked against `_TRANSITIONS` |
| Why it stopped | `finish_reason: str` plus `error` | one `finished_reason` object: `FINISH_MATCHED_TOKEN`, `FINISH_LENGTH`, `FINISH_ERROR` |
| Who decides it is done | `Scheduler.process_batch_result` | `Req.update_finish_state` |
| EOS ids | held by `Scheduler` | carried by each `Req` |
| HTTP `finish_reason` | `"stop"` / `"length"` | `{"type": "stop", "matched": ...}` / `{"type": "length", "length": ...}` |

Each later feature is one more edge in `_TRANSITIONS`. Abort adds `WAITING/RUNNING -> aborted`. Retract under memory pressure adds `RUNNING -> WAITING`.

**Compare with SGLang**: `Req`, `BaseFinishReason`, and the `FINISH_*` classes in `managers/schedule_batch.py`. `Req.check_finished()` is our `update_finish_state`, with more stop conditions (stop strings, regex, invalid token ids). SGLang has no status enum: it infers the state from which list holds the request (`waiting_queue`, `running_batch`) and from `req.finished()`.

## Shutdown

In [9]:
client.close()
server.should_exit = True